<a href="https://colab.research.google.com/github/sunshinebecomesyou/TRUEtalent/blob/main/Report_ZTCA_BJO_SIPOBRI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# ==============================
# BACA DATA
# ==============================

df_divseg = pd.read_excel(
    "/content/1. BRIHC_DivSeg_Map_2026_08.XLSX"
)

df_personel = pd.read_excel(
    "/content/ZTCA_Personel - 09 - To Sina.XLSX",
    sheet_name="Sheet1"
)

tipe_uker = ["UN", "SD", "KW", "KN", "KCP", "KC", "KP"]

output_file = "/content/JobText_ZSTELL_By_UKER.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    # ==============================
    # SHEET OLAH
    # ==============================

    df_personel.to_excel(
        writer,
        sheet_name="Olah",
        index=False
    )

    # ==============================
    # SHEET UKER
    # ==============================

    for tipe in tipe_uker:

        # ------------------------------
        # FILTER DF_DIVSEG
        # ------------------------------

        if tipe == "KN":
            branch_type = "RAO"
        else:
            branch_type = tipe

        divseg_filter = df_divseg[
            df_divseg["Branch Type"]
            .astype(str)
            .str.strip()
            .eq(branch_type)
        ][["Job Text"]].copy()

        # Hapus duplikat Job Text
        divseg_filter = (
            divseg_filter
            .drop_duplicates(subset=["Job Text"])
            .reset_index(drop=True)
        )

        # ------------------------------
        # FILTER DF_PERSONEL
        # ------------------------------

        personel_filter = df_personel[
            df_personel["ZTIPE_UKER"]
            .astype(str)
            .str.strip()
            .eq(tipe)
        ][["ZSTELL_TX"]].copy()

        # Hapus duplikat ZSTELL_TX
        personel_filter = (
            personel_filter
            .drop_duplicates(subset=["ZSTELL_TX"])
            .reset_index(drop=True)
        )

        # ------------------------------
        # GABUNG 2 KOLOM
        # ------------------------------

        hasil = pd.concat(
            [
                divseg_filter,
                personel_filter
            ],
            axis=1
        )

        # ------------------------------
        # SIMPAN
        # ------------------------------

        hasil.to_excel(
            writer,
            sheet_name=tipe,
            index=False
        )

print(f"File berhasil dibuat: {output_file}")

File berhasil dibuat: /content/JobText_ZSTELL_By_UKER.xlsx


In [ ]:
import pandas as pd
from openpyxl import load_workbook

# =========================================================
# FILE
# =========================================================

file_path = "/content/JobText_ZSTELL_By_UKER.xlsx"

# =========================================================
# BACA SOURCE
# =========================================================

df_divseg = pd.read_excel(
    "/content/1. BRIHC_DivSeg_Map_2026_08.XLSX"
)

df_personel = pd.read_excel(
    "/content/ZTCA_Personel - 09 - To Sina.XLSX",
    sheet_name="Sheet1"
)

# =========================================================
# SHEET YANG SUDAH ADA
# =========================================================

tipe_uker = ["UN", "SD", "KW", "KN", "KCP", "KC"]

data_sheets = {}
summary_data = []

# =========================================================
# PROSES SHEET JOB
# =========================================================

for tipe in tipe_uker:

    df = pd.read_excel(
        file_path,
        sheet_name=tipe
    )

    # Ambil seluruh Job Text
    job_text_set = set(
        df["Job Text"]
        .dropna()
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # Cek ZSTELL_TX ke seluruh Job Text
    def cek_data_na(x):

        if pd.isna(x) or str(x).strip() == "":
            return ""

        nilai = str(x).strip()

        if nilai.upper() not in job_text_set:
            return nilai

        return ""

    df["Data NA"] = df["ZSTELL_TX"].apply(cek_data_na)

    # Hitung Data NA
    jumlah_na = (
        df["Data NA"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .sum()
    )

    summary_data.append({
        "ZTIPE_UKER": tipe,
        "Jenis": "JOB",
        "Jumlah Data NA": jumlah_na
    })

    data_sheets[tipe] = df


# =========================================================
# KP JOB
# =========================================================

df_kp_job = pd.read_excel(
    file_path,
    sheet_name="KP"
)

job_text_set_kp = set(
    df_kp_job["Job Text"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)

def cek_kp_job(x):

    if pd.isna(x) or str(x).strip() == "":
        return ""

    nilai = str(x).strip()

    if nilai.upper() not in job_text_set_kp:
        return nilai

    return ""

df_kp_job["Data NA"] = df_kp_job["ZSTELL_TX"].apply(cek_kp_job)

jumlah_kp_job_na = (
    df_kp_job["Data NA"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

summary_data.append({
    "ZTIPE_UKER": "KP",
    "Jenis": "JOB",
    "Jumlah Data NA": jumlah_kp_job_na
})

data_sheets["KP JOB"] = df_kp_job


# =========================================================
# KP AREA
# =========================================================

# ---------------------------------------------------------
# 1. Ambil Working Area dari DivSeg
#    Branch Type tetap KP
# ---------------------------------------------------------

df_kp_area_divseg = (
    df_divseg[
        df_divseg["Branch Type"]
        .astype(str)
        .str.strip()
        .eq("KP")
    ][["Working Area"]]
    .drop_duplicates(subset=["Working Area"])
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 2. Ambil ZADD_AREA dari Personel
#    ZTIPE_UKER = KP
# ---------------------------------------------------------

df_kp_area_personel = (
    df_personel[
        df_personel["ZTIPE_UKER"]
        .astype(str)
        .str.strip()
        .eq("KP")
    ][["ZADD_AREA"]]
    .drop_duplicates(subset=["ZADD_AREA"])
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 3. Gabungkan menjadi 2 kolom
# ---------------------------------------------------------

df_kp_area = pd.concat(
    [
        df_kp_area_divseg,
        df_kp_area_personel
    ],
    axis=1
)


# ---------------------------------------------------------
# 4. Buat set seluruh Working Area
# ---------------------------------------------------------

working_area_set = set(
    df_kp_area["Working Area"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
)


# ---------------------------------------------------------
# 5. Cek ZADD_AREA ke SELURUH Working Area
# ---------------------------------------------------------

def cek_kp_area(x):

    if pd.isna(x) or str(x).strip() == "":
        return ""

    nilai = str(x).strip()

    # Tidak ditemukan di seluruh Working Area
    if nilai.upper() not in working_area_set:
        return nilai

    return ""


df_kp_area["Data NA"] = (
    df_kp_area["ZADD_AREA"]
    .apply(cek_kp_area)
)


# ---------------------------------------------------------
# 6. Hitung jumlah Data NA KP AREA
# ---------------------------------------------------------

jumlah_kp_area_na = (
    df_kp_area["Data NA"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

summary_data.append({
    "ZTIPE_UKER": "KP",
    "Jenis": "AREA",
    "Jumlah Data NA": jumlah_kp_area_na
})


# =========================================================
# SUMMARY
# =========================================================

df_summary = pd.DataFrame(summary_data)


# =========================================================
# TULIS KEMBALI KE FILE YANG SAMA
# =========================================================

with pd.ExcelWriter(
    file_path,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:

    # Summary
    df_summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    # Sheet Job
    for tipe, df in data_sheets.items():

        df.to_excel(
            writer,
            sheet_name=tipe,
            index=False
        )

    # KP AREA
    df_kp_area.to_excel(
        writer,
        sheet_name="KP AREA",
        index=False
    )


# =========================================================
# ATUR URUTAN SHEET
# =========================================================

wb = load_workbook(file_path)

urutan_sheet = [
    "Summary",
    "Olah",
    "UN",
    "SD",
    "KW",
    "KN",
    "KCP",
    "KC",
    "KP JOB",
    "KP AREA"
]

# Ambil sheet yang memang ada
sheet_tersedia = [
    wb[sheet]
    for sheet in urutan_sheet
    if sheet in wb.sheetnames
]

wb._sheets = sheet_tersedia

wb.save(file_path)


# =========================================================
# HASIL
# =========================================================

print("Selesai!")
print(f"File diperbarui: {file_path}")
print()
print("Summary:")
print(df_summary)

Selesai!
File diperbarui: /content/JobText_ZSTELL_By_UKER.xlsx

Summary:
  ZTIPE_UKER Jenis  Jumlah Data NA
0         UN   JOB               0
1         SD   JOB               0
2         KW   JOB              12
3         KN   JOB               2
4        KCP   JOB               2
5         KC   JOB               1
6         KP   JOB              14
7         KP  AREA               2


In [ ]:
import pandas as pd
from openpyxl import load_workbook

# =========================================================
# FILE
# =========================================================

file_path = "/content/JobText_ZSTELL_By_UKER.xlsx"

# =========================================================
# SHEET YANG AKAN DICEK
# =========================================================

sheet_list = [
    "UN",
    "SD",
    "KW",
    "KN",
    "KCP",
    "KC",
    "KP JOB",
    "KP AREA"
]

data_na_all = []

# =========================================================
# AMBIL SEMUA DATA NA
# =========================================================

for sheet in sheet_list:

    df = pd.read_excel(
        file_path,
        sheet_name=sheet
    )

    # Pastikan sheet memiliki kolom Data NA
    if "Data NA" not in df.columns:
        continue

    # Ambil hanya Data NA yang tidak kosong
    df_na = df[
        df["Data NA"].notna()
        & df["Data NA"].astype(str).str.strip().ne("")
    ].copy()

    if df_na.empty:
        continue

    # Tentukan jenis data
    if sheet == "KP AREA":
        jenis = "AREA"
    else:
        jenis = "JOB"

    # Buat hasil
    hasil = pd.DataFrame({
        "Sheet": sheet,
        "Jenis": jenis,
        "Data NA": df_na["Data NA"].astype(str).str.strip()
    })

    data_na_all.append(hasil)


# =========================================================
# GABUNGKAN SEMUA DATA NA
# =========================================================

if data_na_all:

    df_data_na_all = pd.concat(
        data_na_all,
        ignore_index=True
    )

else:

    df_data_na_all = pd.DataFrame(
        columns=[
            "Sheet",
            "Jenis",
            "Data NA"
        ]
    )


# =========================================================
# HAPUS SHEET LAMA JIKA ADA
# =========================================================

wb = load_workbook(file_path)

if "Data NA All" in wb.sheetnames:
    del wb["Data NA All"]

wb.save(file_path)


# =========================================================
# TAMBAHKAN SHEET DATA NA ALL
# =========================================================

with pd.ExcelWriter(
    file_path,
    engine="openpyxl",
    mode="a"
) as writer:

    df_data_na_all.to_excel(
        writer,
        sheet_name="Data NA All",
        index=False
    )


# =========================================================
# PINDAHKAN DATA NA ALL KE PALING AKHIR
# =========================================================

wb = load_workbook(file_path)

ws = wb["Data NA All"]

wb._sheets.remove(ws)
wb._sheets.append(ws)

wb.save(file_path)


# =========================================================
# HASIL
# =========================================================

print("Selesai!")
print(f"File diperbarui: {file_path}")
print(f"Total Data NA: {len(df_data_na_all)}")

Selesai!
File diperbarui: /content/JobText_ZSTELL_By_UKER.xlsx
Total Data NA: 33
